# RutaPro · Notebook 00: Construcción del dataset

**Proyecto:** Sistema de recomendación de cursos profesionales para estudiantes universitarios
**Fase 2 del trabajo:** construcción y documentación del dataset

Este notebook genera las tres entidades del proyecto:

| Entidad | Archivo | Origen |
|---|---|---|
| Cursos | `01_Datos/cursos.csv` | **Real**: web scraping de Coursera, edX, Platzi y DataCamp (19/10/2026) |
| Usuarios | `01_Datos/usuarios.csv` | **Simulado** con reglas explícitas |
| Interacciones | `01_Datos/interacciones.csv` | **Simulado** con reglas explícitas |

> **Declaración de datos simulados.** RutaPro es una organización ficticia y no existen registros reales de estudiantes. Los usuarios y sus interacciones se generan con las reglas descritas en la sección 4, con semilla fija. El catálogo de cursos sí corresponde a cursos reales publicados en Coursera, edX, Platzi y DataCamp. Como RutaPro está dirigido a estudiantes peruanos, el catálogo se construye con **aproximadamente la mitad de cursos en español**.

**Estructura del notebook**
1. Configuración
2. Web scraping del catálogo (código reproducible y datos capturados)
3. Limpieza y curación del catálogo
4. Simulación de usuarios
5. Simulación de interacciones
6. Validaciones de integridad
7. Exportación

In [ ]:
import os, sys, subprocess
from google.colab import drive

NECESITA_LIGHTFM = False

# 1) Conectar el Drive y entrar a la carpeta del proyecto
drive.mount('/content/drive')
PROY = "/content/drive/MyDrive/RutaPro"
os.chdir(PROY + "/02_Notebooks")
print("Carpeta actual:", os.getcwd())

# 2) Comprobar que están los archivos imprescindibles
faltan = [f for f in ["rutapro_rec.py", "rutapro_sim.py", "rutapro_extra.py", "rutapro_mooc.py",
                      "resultados/hiperparametros.json", "../01_Datos/cursos.csv",
                      "../01_Datos/pares_usuario_curso.csv"] if not os.path.exists(f)]
print("Archivos que faltan:", faltan if faltan else "ninguno, todo en su sitio")

# 3) LightFM: usa la copia compilada en tu Drive; si no existe, la compila (≈ 2 min) y la guarda
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("Falló: " + cmd[:70])

if NECESITA_LIGHTFM:
    LF = PROY + "/lightfm_src"
    if not os.path.exists(LF + "/lightfm"):
        print("Compilando LightFM, un momento...")
        sh("pip install -q cython")
        sh("rm -rf /content/lightfm_src && git clone -q --depth 1 https://github.com/lyst/lightfm.git /content/lightfm_src")
        p = "/content/lightfm_src/setup.py"
        s = open(p).read()
        s = s.replace("compiler_directives={'language_level' : \"3\"}",
                      "compiler_directives={'language_level': \"3\", 'legacy_implicit_noexcept': True}").replace("-march=native", "-O2")
        open(p, "w").write(s)
        sh("cd /content/lightfm_src && python setup.py cythonize -q && python setup.py build_ext --inplace -q")
        sh("rm -rf '" + LF + "' && cp -r /content/lightfm_src '" + LF + "'")
    sys.path.insert(0, LF)
    import lightfm
    print("LightFM", lightfm.__version__)

In [ ]:
import re, json, time, math
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

# Rutas: el notebook vive en 02_Notebooks/ y los datos en 01_Datos/
BASE = Path.cwd().parent if Path.cwd().name == "02_Notebooks" else Path.cwd()
DATA = BASE / "01_Datos"
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

FECHA_EXTRACCION = "2026-10-19"
FECHA_CORTE = pd.Timestamp("2026-10-15")   # último día con eventos registrados
pd.set_option("display.max_colwidth", 60)
print("Carpeta de datos:", DATA)

Carpeta de datos: /content/01_Datos


## 2. Web scraping del catálogo

### 2.1 Estrategia
- **Fuentes en inglés:** páginas públicas de búsqueda de Coursera (`/courses?query=...`).
- **Fuentes en español:** temas de edX en español (`/es/learn/...`), rutas y fichas de cursos de **Platzi** (`platzi.com/escuela/datos`, `platzi.com/ruta/...`, `platzi.com/cursos/...`), fichas de cursos en español de **DataCamp** (`datacamp.com/es/courses/...`) y fichas de Coursera en español. Platzi y DataCamp se incluyen por ser plataformas muy usadas en Latinoamérica. Udemy se descartó porque su `robots.txt` no permite la extracción automática, y Crehana porque su listado se carga dinámicamente y no expone los cursos en el HTML.
- **Consultas:** `sql`, `excel`, `power bi`, `tableau`, `statistics`, `machine learning`, `r programming`, `python`, `project management`, `communication skills` y búsquedas de cursos en español.
- **Campos extraídos:** título, institución o docente, URL, tipo de programa, nivel, duración, rating, número de reseñas u opiniones y habilidades o temas que declara la plataforma (en Platzi y DataCamp, los temas del temario).
- **Buenas prácticas:** se revisa `robots.txt` antes de cada petición, se espera entre peticiones y se usa un *User-Agent* identificable. La API de Coursera (`api.coursera.org`) **no se usa** porque su `robots.txt` la excluye.

### 2.2 Reproducibilidad
La extracción original se hizo el 19/10/2026 y quedó guardada en `01_Datos/raw/cursos_scraping_raw.txt`. Las páginas de Coursera y edX cambian con frecuencia (orden, diseño HTML), por lo que **el resto del notebook trabaja sobre esa captura**. La función de abajo permite repetir la extracción: basta con poner `EJECUTAR_SCRAPING = True`. Si el HTML de las páginas cambió y la función no encuentra cursos, el notebook continúa con la captura guardada.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib import robotparser
from urllib.parse import urljoin, urlparse, quote

EJECUTAR_SCRAPING = False   # cambiar a True para volver a extraer desde la web
HEADERS = {"User-Agent": "RutaPro-academic-scraper/1.0 (proyecto universitario)"}
PAUSA_SEG = 3

_robots = {}
def permitido(url):
    # Consulta robots.txt del dominio antes de pedir la página
    dom = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    if dom not in _robots:
        rp = robotparser.RobotFileParser(); rp.set_url(dom + "/robots.txt")
        try: rp.read()
        except Exception: return False
        _robots[dom] = rp
    return _robots[dom].can_fetch(HEADERS["User-Agent"], url)

def get_html(url):
    if not permitido(url):
        print("robots.txt no permite:", url); return None
    time.sleep(PAUSA_SEG)
    r = requests.get(url, headers=HEADERS, timeout=30)
    return r.text if r.ok else None

PREFIJOS_COURSERA = ("/learn/", "/specializations/", "/professional-certificates/", "/projects/")

def urls_busqueda_coursera(query):
    # Paso 1: obtiene las URLs de los cursos que aparecen en la página de búsqueda
    html = get_html(f"https://www.coursera.org/courses?query={quote(query)}")
    if not html: return []
    soup = BeautifulSoup(html, "html.parser")
    urls = {urljoin("https://www.coursera.org", a["href"].split("?")[0])
            for a in soup.find_all("a", href=True) if a["href"].startswith(PREFIJOS_COURSERA)}
    return sorted(urls)

def ficha_coursera(url):
    # Paso 2: lee la ficha del curso; Coursera publica nombre, proveedor y rating en JSON-LD
    html = get_html(url)
    if not html: return None
    soup = BeautifulSoup(html, "html.parser")
    reg = {"plataforma": "Coursera", "url": url}
    for s in soup.find_all("script", type="application/ld+json"):
        try: data = json.loads(s.string or "")
        except Exception: continue
        for d in (data if isinstance(data, list) else data.get("@graph", [data])):
            if d.get("@type") in ("Course", "Product"):
                reg["titulo"] = d.get("name")
                prov = d.get("provider") or d.get("brand") or {}
                reg["institucion"] = prov.get("name") if isinstance(prov, dict) else None
                ar = d.get("aggregateRating") or {}
                reg["rating"] = ar.get("ratingValue"); reg["n_resenas"] = ar.get("ratingCount") or ar.get("reviewCount")
    texto = soup.get_text(" ", strip=True)
    m = re.search(r"\b(Beginner|Intermediate|Advanced|Mixed) level\b", texto)
    reg["nivel"] = m.group(1) if m else None
    return reg if reg.get("titulo") else None

def scrapear_coursera(queries, max_por_query=12):
    filas = []
    for q in queries:
        for u in urls_busqueda_coursera(q)[:max_por_query]:
            f = ficha_coursera(u)
            if f: f["query"] = q; filas.append(f)
    return pd.DataFrame(filas)

if EJECUTAR_SCRAPING:
    QUERIES = ["sql", "excel", "power bi", "tableau", "statistics", "machine learning",
               "r programming", "python", "project management", "communication skills"]
    df_live = scrapear_coursera(QUERIES)
    print("Cursos extraídos en vivo:", len(df_live))
    if len(df_live):
        df_live.assign(fecha_extraccion=datetime.today().date()).to_csv(RAW / "cursos_scraping_live.csv", index=False)
else:
    print("Se usa la captura del", FECHA_EXTRACCION, "guardada en raw/cursos_scraping_raw.txt")

Se usa la captura del 2026-10-19 guardada en raw/cursos_scraping_raw.txt


In [ ]:
raw = pd.read_csv(RAW / "cursos_scraping_raw.txt", sep=r"\s\|\s", engine="python", na_values=["NA"])
raw["fecha_extraccion"] = FECHA_EXTRACCION
print("Registros capturados:", raw.shape[0])
print(raw.groupby(["idioma", "plataforma"]).size().rename("n"))
raw.head()

Registros capturados: 171
idioma  plataforma
en      Coursera      116
es      Coursera        3
        DataCamp        9
        Platzi         35
        edX             8
Name: n, dtype: int64


,plataforma,query,idioma,titulo,institucion,url,tipo,nivel,duracion,rating,n_resenas,habilidades,fecha_extraccion
0,Coursera,sql,en,Databases and SQL for Data Science with Python,IBM,https://www.coursera.org/learn/sql-data-science,Course,Beginner,1 - 3 Months,4.7,23K,SQL; Database Management; Relational Databases; Stored P...,2026-10-19
1,Coursera,sql,en,SQL Foundations,Microsoft,https://www.coursera.org/learn/sql-foundations,Course,Beginner,1 - 4 Weeks,4.5,291,SQL; Microsoft SQL Servers; Relational Databases; Transa...,2026-10-19
2,Coursera,sql,en,SQL for Data Science,"University of California, Davis",https://www.coursera.org/learn/sql-for-data-science,Course,Beginner,1 - 4 Weeks,4.6,17K,Data Governance; SQL; Data Quality; Data Access; Data Ma...,2026-10-19
3,Coursera,sql,en,Microsoft SQL Server,Microsoft,https://www.coursera.org/professional-certificates/micro...,Professional Certificate,Beginner,3 - 6 Months,4.5,365,Star Schema; Microsoft SQL Servers; Relational Databases...,2026-10-19
4,Coursera,sql,en,SQL: A Practical Introduction for Querying Databases,IBM,https://www.coursera.org/learn/sql-practical-introductio...,Course,Beginner,1 - 3 Months,4.7,721,SQL; Relational Databases; Microsoft SQL Servers; MySQL;...,2026-10-19


## 3. Limpieza y curación del catálogo

Reglas aplicadas, en este orden:

1. **Duplicados:** un mismo curso aparece en varias búsquedas (por ejemplo, *Google Data Analytics*). Se conserva una fila por URL.
2. **Pertinencia:** se excluyen cursos que no corresponden al público de RutaPro (Java, ciberseguridad, automatización de TI, construcción, Microsoft 365, cursos de idioma inglés, herramientas como Genial.ly o Power Virtual Agents).
3. **Categoría:** se asigna por palabras clave del título (en inglés y en español), en orden de prioridad (ver `reglas_categoria`). Si ninguna aplica, el curso queda en "Análisis de Datos".
4. **Variables numéricas:** "23K" → 23 000 reseñas; la duración en texto se convierte a horas aproximadas.
5. **Valores no publicados:** los cursos de edX no muestran nivel, duración, rating ni reseñas en el listado, y DataCamp publica el número de reseñas pero no un rating promedio numérico. El nivel se infiere del título, la duración se imputa con la mediana de su categoría, el rating con la mediana del catálogo y las reseñas con el percentil 25 (valor conservador). Cada imputación queda marcada en una columna `*_imputado`.
6. **Tamaño y balance de idioma:** como máximo 5 cursos en inglés por categoría y **todos** los cursos en español. Dentro de cada categoría se incluye primero el curso más reseñado de cada nivel de dificultad disponible (para que existan rutas Básico → Intermedio → Avanzado) y luego se completa con los de más reseñas.

In [ ]:
cat = raw.drop_duplicates("url").copy()

excluir = r"Java|Cybersecurity|IT Automation|Construction|Microsoft 365|English|Small Talk|Genial\.ly|Virtual Agents"
cat = cat[~cat["titulo"].str.contains(excluir, case=False, regex=True)]

# Reglas de categoría (orden = prioridad)
reglas_categoria = [
    ("Gestión de Proyectos", r"project|agile|proyecto|scrum"),
    ("R",                    r"\bR\b(?! &)|with R|con R|en R\b|\bR Programming|[Tt]idyverse"),
    ("Estadística",          r"statistic|estad[ií]stic|probabilit|probabilidad"),
    ("Machine Learning",     r"machine learning|deep learning|mlops|redes neuronales|aprendizaje supervisado|regresi[oó]n lineal con|decision trees"),
    ("Tableau y Visualización", r"tableau|looker|visualizaci[oó]n de datos para bi"),
    ("Power BI",             r"power ?bi"),
    ("Excel",                r"excel"),
    ("SQL y Bases de Datos", r"\bsql\b|database|query|bases de datos|postgresql"),
    ("Python",               r"python|pandas"),
    ("Habilidades Blandas",  r"communic|speaking|interpersonal|soft skills|listening|storytelling|comunicaci|oratoria|habilidades blandas|feedback"),
    ("Análisis de Datos",    r"data analy|an[aá]lisis exploratorio|ciencia de datos|business intelligence|decisiones basadas"),
]
def asignar_categoria(row):
    for nombre, patron in reglas_categoria:
        flags = 0 if nombre == "R" else re.IGNORECASE
        if re.search(patron, row["titulo"], flags):
            return nombre
    return "Análisis de Datos"
cat["categoria"] = cat.apply(asignar_categoria, axis=1)
cat["categoria"].value_counts()

,count
categoria,
Python,19
Machine Learning,18
Estadística,16
R,16
Gestión de Proyectos,16
Excel,16
Power BI,15
Tableau y Visualización,14
Habilidades Blandas,13


In [ ]:
def a_numero(x):
    if pd.isna(x): return np.nan
    s = str(x).replace(",", "").strip().upper()
    mult = 1000 if s.endswith("K") else 1
    return float(s.rstrip("K")) * mult

def a_horas(txt):
    # Convierte la duración publicada a horas aproximadas (supuesto Coursera: ~10 h por semana)
    if pd.isna(txt): return np.nan
    t = str(txt).lower()
    m = re.search(r"(\d+)\s*semanas?\s*a\s*(\d+)\s*horas", t)
    if m: return float(m.group(1)) * float(m.group(2))
    m = re.match(r"^(?:(\d+)\s*(?:h|horas?)\b)?\s*(?:(\d+)\s*min)?", t)   # Platzi / DataCamp: "3 horas", "3 h 30 min", "45 min"
    if m and (m.group(1) or m.group(2)):
        return float(m.group(1) or 0) + float(m.group(2) or 0) / 60
    mapa = {"less than 2 hours": 2, "1 - 4 weeks": 20, "1 - 3 months": 45, "3 - 6 months": 110}
    return float(mapa.get(t, np.nan))

TIPO = {"Course": "Curso", "Specialization": "Especialización",
        "Professional Certificate": "Certificado profesional", "Guided Project": "Proyecto guiado"}
NIVEL = {"Beginner": "Básico", "Intermediate": "Intermedio", "Advanced": "Avanzado", "Mixed": "Intermedio",
         "Básico": "Básico", "Intermedio": "Intermedio", "Avanzado": "Avanzado"}

cat["n_resenas"] = cat["n_resenas"].map(a_numero)
cat["duracion_horas"] = cat["duracion"].map(a_horas)
cat["tipo"] = cat["tipo"].map(TIPO)
cat["dificultad"] = cat["nivel"].map(NIVEL)

# --- Imputaciones (solo edX en español) ---
cat["dificultad_imputada"] = cat["dificultad"].isna()
def nivel_por_titulo(t):
    t = t.lower()
    if "avanzado" in t: return "Avanzado"
    if any(k in t for k in ["básico", "basico", "fundamentos", "aprender a programar", "conceptos básicos"]): return "Básico"
    return "Intermedio"
cat.loc[cat["dificultad_imputada"], "dificultad"] = cat.loc[cat["dificultad_imputada"], "titulo"].map(nivel_por_titulo)

cat["duracion_imputada"] = cat["duracion_horas"].isna()
cat["duracion_horas"] = cat["duracion_horas"].fillna(cat.groupby("categoria")["duracion_horas"].transform("median"))
cat["duracion_horas"] = cat["duracion_horas"].fillna(cat["duracion_horas"].median())

cat["rating_imputado"] = cat["rating"].isna()
cat["rating"] = cat["rating"].fillna(cat["rating"].median())
cat["n_resenas_imputado"] = cat["n_resenas"].isna()
cat["n_resenas"] = cat["n_resenas"].fillna(cat["n_resenas"].quantile(0.25))

# Las habilidades de los cursos de edX se derivan del título (edX no las muestra en el listado)
cat["habilidades_fuente"] = np.select([cat["plataforma"].eq("edX"), cat["plataforma"].isin(["Platzi", "DataCamp"])],
                                     ["inferidas del título", "temario publicado por la plataforma"],
                                     "skills publicadas por la plataforma")

cat[["dificultad_imputada", "duracion_imputada", "rating_imputado", "n_resenas_imputado"]].sum()

,0
dificultad_imputada,8
duracion_imputada,8
rating_imputado,17
n_resenas_imputado,8


In [ ]:
# Selección final: todos los cursos en español + hasta 5 en inglés por categoría → catálogo ~50 % / 50 %
MAX_EN_POR_CATEGORIA = 5
es = cat[cat["idioma"] == "es"]
def seleccionar(g, k=MAX_EN_POR_CATEGORIA):
    # 1) el curso más reseñado de cada nivel de dificultad disponible (asegura rutas Básico→Avanzado)
    # 2) se completa hasta k con los cursos de más reseñas
    g = g.sort_values("n_resenas", ascending=False)
    base = g.groupby("dificultad").head(1)
    resto = g.drop(base.index).head(max(0, k - len(base)))
    return pd.concat([base, resto])
en = pd.concat([seleccionar(g) for _, g in cat[cat["idioma"] == "en"].groupby("categoria")])
cursos = pd.concat([en, es]).sort_values(["categoria", "dificultad", "titulo"]).reset_index(drop=True)
cursos.insert(0, "course_id", [f"C{i:03d}" for i in range(1, len(cursos) + 1)])

cursos["habilidades"] = cursos["habilidades"].str.replace(r"\s*;\s*", "; ", regex=True)
cursos = cursos[["course_id", "titulo", "plataforma", "institucion", "idioma", "categoria", "tipo", "dificultad",
                 "duracion_horas", "rating", "n_resenas", "habilidades", "url", "fecha_extraccion",
                 "dificultad_imputada", "duracion_imputada", "rating_imputado", "n_resenas_imputado",
                 "habilidades_fuente"]]
print("Cursos en el catálogo final:", len(cursos))
display(pd.crosstab(cursos["categoria"], cursos["idioma"], margins=True))
display(pd.crosstab(cursos["plataforma"], cursos["idioma"], margins=True))
display(pd.crosstab(cursos["categoria"], cursos["dificultad"]))

Cursos en el catálogo final: 107


idioma,en,es,All
categoria,,,
Análisis de Datos,2,4,6
Estadística,5,4,9
Excel,5,7,12
Gestión de Proyectos,5,5,10
Habilidades Blandas,5,5,10
Machine Learning,5,7,12
Power BI,5,4,9
Python,5,7,12
R,5,6,11


idioma,en,es,All
plataforma,,,
Coursera,52,3,55
DataCamp,0,9,9
Platzi,0,35,35
edX,0,8,8
All,52,55,107


dificultad,Avanzado,Básico,Intermedio
categoria,,,
Análisis de Datos,0,4,2
Estadística,1,7,1
Excel,2,6,4
Gestión de Proyectos,0,6,4
Habilidades Blandas,1,8,1
Machine Learning,3,4,5
Power BI,1,6,2
Python,0,9,3
R,0,7,4


## 4. Simulación de usuarios

Se generan **150 estudiantes** con las siguientes reglas:

| Variable | Regla |
|---|---|
| `carrera` | Distribución fija: Estadística e Informática 25 %, Economía 20 %, Ing. Industrial 20 %, Ing. de Sistemas 15 %, Administración 20 % |
| `ciclo` | Entero uniforme entre 5 y 10 |
| `objetivo_laboral` | Depende de la carrera (matriz `P_OBJETIVO`) |
| `nivel` | Depende del ciclo: ciclos 5-6 tienden a Básico, 7-8 a Intermedio, 9-10 a Avanzado (con ruido) |
| `idioma_preferido` | 40 % solo español, 60 % español e inglés |
| `intereses` | 2 o 3 categorías, muestreadas según la afinidad de su objetivo laboral |
| `habilidades_actuales` | 1 a 3 categorías que ya domina a nivel básico, ligadas a su carrera |
| `fecha_registro` | Uniforme entre el 01/01/2026 y el 31/08/2026 |

La **afinidad objetivo-categoría** (`AFINIDAD`) resume qué tan útil es cada categoría para cada puesto. Es un supuesto del equipo basado en los requisitos típicos de avisos de prácticas para esos puestos, y es la regla principal que guía la simulación.

In [ ]:
CATS = ["Python", "SQL y Bases de Datos", "Excel", "Power BI", "Tableau y Visualización", "Estadística", "R",
        "Machine Learning", "Análisis de Datos", "Gestión de Proyectos", "Habilidades Blandas"]

AFINIDAD = pd.DataFrame({
    #                      Py   SQL  Exc  PBI  Tab  Est  R    ML   AD   GP   HB
    "Data Analyst":       [.9, 1.0, .9,  .8,  .7,  .8,  .5,  .3,  1.0, .2,  .5],
    "BI Analyst":         [.4, 1.0, .9,  1.0, 1.0, .5,  .2,  .2,  .8,  .3,  .5],
    "Data Scientist":     [1.0, .7, .3,  .3,  .3,  1.0, .7,  1.0, .7,  .1,  .3],
    "Analista Financiero":[.5, .6,  1.0, .8,  .5,  .6,  .3,  .3,  .7,  .4,  .5],
    "Marketing Analyst":  [.5, .6,  .8,  .7,  .8,  .7,  .4,  .4,  .9,  .3,  .7],
    "Project Analyst":    [.2, .4,  .8,  .6,  .4,  .3,  .1,  .1,  .5,  1.0, 1.0],
}, index=CATS).T

CARRERAS = {"Estadística e Informática": .25, "Economía": .20, "Ingeniería Industrial": .20,
            "Ingeniería de Sistemas": .15, "Administración": .20}
OBJETIVOS = list(AFINIDAD.index)
P_OBJETIVO = pd.DataFrame({
    #                            DA   BI   DS   AF   MK   PA
    "Estadística e Informática": [.35, .15, .35, .05, .05, .05],
    "Economía":                  [.25, .15, .10, .30, .15, .05],
    "Ingeniería Industrial":     [.25, .25, .05, .10, .05, .30],
    "Ingeniería de Sistemas":    [.25, .25, .35, .00, .00, .15],
    "Administración":            [.15, .15, .00, .25, .25, .20],
}, index=OBJETIVOS).T
BASE_CARRERA = {  # categorías que la carrera suele cubrir en sus cursos obligatorios
    "Estadística e Informática": ["Estadística", "R", "Python", "Excel"],
    "Economía": ["Excel", "Estadística"],
    "Ingeniería Industrial": ["Excel", "Gestión de Proyectos", "Estadística"],
    "Ingeniería de Sistemas": ["Python", "SQL y Bases de Datos"],
    "Administración": ["Excel", "Habilidades Blandas"],
}
display(AFINIDAD)

,Python,SQL y Bases de Datos,Excel,Power BI,Tableau y Visualización,Estadística,R,Machine Learning,Análisis de Datos,Gestión de Proyectos,Habilidades Blandas
Data Analyst,0.9,1.0,0.9,0.8,0.7,0.8,0.5,0.3,1.0,0.2,0.5
BI Analyst,0.4,1.0,0.9,1.0,1.0,0.5,0.2,0.2,0.8,0.3,0.5
Data Scientist,1.0,0.7,0.3,0.3,0.3,1.0,0.7,1.0,0.7,0.1,0.3
Analista Financiero,0.5,0.6,1.0,0.8,0.5,0.6,0.3,0.3,0.7,0.4,0.5
Marketing Analyst,0.5,0.6,0.8,0.7,0.8,0.7,0.4,0.4,0.9,0.3,0.7
Project Analyst,0.2,0.4,0.8,0.6,0.4,0.3,0.1,0.1,0.5,1.0,1.0


In [ ]:
N_USUARIOS = 150
NIVELES = ["Básico", "Intermedio", "Avanzado"]

def simular_usuario(i):
    carrera = rng.choice(list(CARRERAS), p=list(CARRERAS.values()))
    ciclo = int(rng.integers(5, 11))
    objetivo = rng.choice(OBJETIVOS, p=P_OBJETIVO.loc[carrera].values)
    # nivel: el ciclo desplaza la probabilidad hacia niveles más altos
    p_nivel = {5: [.70, .25, .05], 6: [.60, .32, .08], 7: [.40, .45, .15],
               8: [.30, .50, .20], 9: [.20, .50, .30], 10: [.15, .45, .40]}[ciclo]
    nivel = rng.choice(NIVELES, p=p_nivel)
    idioma = rng.choice(["es", "es+en"], p=[.4, .6])
    w = AFINIDAD.loc[objetivo].values ** 2; w = w / w.sum()
    intereses = list(rng.choice(CATS, size=int(rng.integers(2, 4)), replace=False, p=w))
    base = BASE_CARRERA[carrera]
    habilidades = list(rng.choice(base, size=min(len(base), int(rng.integers(1, 4))), replace=False))
    registro = pd.Timestamp("2026-01-01") + pd.Timedelta(days=int(rng.integers(0, 243)))
    return dict(user_id=f"U{i:03d}", carrera=carrera, ciclo=ciclo, nivel=nivel, idioma_preferido=idioma,
                objetivo_laboral=objetivo, intereses="; ".join(intereses),
                habilidades_actuales="; ".join(habilidades), fecha_registro=registro.date())

usuarios = pd.DataFrame([simular_usuario(i) for i in range(1, N_USUARIOS + 1)])
print(usuarios.shape)
display(usuarios.head())
display(pd.crosstab(usuarios["carrera"], usuarios["objetivo_laboral"]))

(150, 9)


,user_id,carrera,ciclo,nivel,idioma_preferido,objetivo_laboral,intereses,habilidades_actuales,fecha_registro
0,U001,Ingeniería de Sistemas,8,Intermedio,es,Project Analyst,Habilidades Blandas; Gestión de Proyectos,Python; SQL y Bases de Datos,2026-04-20
1,U002,Economía,6,Intermedio,es+en,Analista Financiero,Excel; Power BI; SQL y Bases de Datos,Estadística; Excel,2026-06-03
2,U003,Ingeniería de Sistemas,9,Avanzado,es+en,Project Analyst,Excel; Gestión de Proyectos,Python; SQL y Bases de Datos,2026-02-07
3,U004,Ingeniería de Sistemas,10,Intermedio,es,Project Analyst,Gestión de Proyectos; Excel; Análisis de Datos,Python,2026-05-18
4,U005,Economía,9,Avanzado,es,Marketing Analyst,Análisis de Datos; Tableau y Visualización; Power BI,Excel; Estadística,2026-06-04


objetivo_laboral,Analista Financiero,BI Analyst,Data Analyst,Data Scientist,Marketing Analyst,Project Analyst
carrera,,,,,,
Administración,6,2,4,0,11,4
Economía,13,3,3,6,2,3
Estadística e Informática,4,7,18,8,3,1
Ingeniería Industrial,3,6,4,2,1,9
Ingeniería de Sistemas,0,6,6,7,0,8


## 5. Simulación de interacciones

Cada estudiante recorre el catálogo de forma **secuencial en el tiempo**. En cada paso elige un curso con una probabilidad proporcional a un *puntaje de utilidad*:

$$
\text{utilidad}(u,c) = 2.0\cdot \text{afinidad}(\text{objetivo}_u, \text{cat}_c) + 0.6\cdot \mathbb{1}[\text{cat}_c \in \text{intereses}_u] + 1.5\cdot \text{pop}_c + \text{ajuste\_nivel}(u,c) + \text{ajuste\_idioma}(u,c)
$$

- $\text{pop}_c$: percentil del número de reseñas del curso **dentro de su plataforma** (0 a 1). Se usa el percentil por plataforma porque las reseñas de Coursera, Platzi y DataCamp no son comparables en valor absoluto.
- **Ajuste de nivel (asimétrico):** +0.4 si la dificultad del curso coincide con el nivel *efectivo* del estudiante; −0.5 si el curso está un nivel por encima y −1.5 si está dos niveles por encima; −0.2 y −0.6 si está uno o dos niveles por debajo (repasar es menos costoso que saltarse niveles). El nivel efectivo **sube** cada vez que el estudiante completa 2 cursos, lo que genera rutas Básico → Intermedio → Avanzado.
- **Prerrequisito:** un curso Avanzado de una categoría tiene peso ×0.3 si el estudiante aún no ha tomado nada de esa categoría.
- **Ajuste de idioma:** −1.5 a los cursos en inglés para estudiantes que solo leen español.
- **Ya domina:** −0.5 a cursos básicos de categorías que el estudiante declara dominar.
- **Exploración:** con probabilidad 0.15 el curso se elige al azar (ruido realista).
- Probabilidad de elegir: $P(c) \propto \exp(\text{utilidad}/T)$, con temperatura $T = 0.6$, sin repetir cursos.

**Embudo de eventos por curso elegido** (cada etapa depende de la anterior):

| Evento | Peso | Probabilidad |
|---|---|---|
| `vista` | 1 | siempre |
| `favorito` | 2 | 0.25 |
| `inscripcion` | 3 | 0.30 + 0.35·(utilidad normalizada) |
| `finalizacion` | 4 | 0.45 + 0.25·[nivel coincide] − 0.15·[curso de más de 100 h] |
| calificación (1-5) | n.a. | solo al finalizar: rating real del curso ± ruido, −1 si el nivel no encajó |

**Tiempos:** el primer curso se mira entre 0 y 20 días después del registro; entre un curso y el siguiente pasan en promedio 6 días (distribución exponencial). La finalización ocurre después de `duracion_horas / 6` semanas aprox. Los eventos posteriores al 15/10/2026 no se registran (censura natural del periodo de observación).

**Número de cursos por estudiante:** binomial negativa con media ≈ 13, mínimo 3. Esto produce usuarios muy activos y otros poco activos, como en plataformas reales.

In [ ]:
cursos_idx = cursos.set_index("course_id")
NIV_NUM = {"Básico": 0, "Intermedio": 1, "Avanzado": 2}
pop = cursos_idx.groupby("plataforma")["n_resenas"].rank(pct=True)   # popularidad relativa dentro de cada plataforma
T, P_EXPLORA = 0.6, 0.15

def utilidad(u, c_id, nivel_ef, cats_tomadas):
    c = cursos_idx.loc[c_id]
    s = 2.0 * AFINIDAD.loc[u.objetivo_laboral, c.categoria]
    s += 0.6 * (c.categoria in u.intereses.split("; "))
    s += 1.5 * pop[c_id]
    gap = NIV_NUM[c.dificultad] - nivel_ef          # >0: curso más difícil que el nivel del estudiante
    s += {-2: -0.6, -1: -0.2, 0: 0.4, 1: -0.5, 2: -1.5}[gap]
    if u.idioma_preferido == "es" and c.idioma == "en": s -= 1.5
    if c.categoria in u.habilidades_actuales.split("; ") and c.dificultad == "Básico": s -= 0.5
    return s

def simular_interacciones_usuario(u):
    eventos = []
    n_cursos = max(3, int(rng.negative_binomial(4, 4 / (4 + 13))))
    nivel_ef = NIV_NUM[u.nivel]
    tomados, cats_tomadas, completados = set(), set(), 0
    t = pd.Timestamp(u.fecha_registro) + pd.Timedelta(days=int(rng.integers(0, 21)), hours=int(rng.integers(8, 23)))
    for _ in range(n_cursos):
        disponibles = [c for c in cursos_idx.index if c not in tomados]
        if rng.random() < P_EXPLORA:
            c_id = rng.choice(disponibles)
        else:
            s = np.array([utilidad(u, c, nivel_ef, cats_tomadas) for c in disponibles])
            prereq = np.array([0.3 if (cursos_idx.at[c, "dificultad"] == "Avanzado" and
                                       cursos_idx.at[c, "categoria"] not in cats_tomadas) else 1.0
                               for c in disponibles])
            w = np.exp((s - s.max()) / T) * prereq
            c_id = rng.choice(disponibles, p=w / w.sum())
        tomados.add(c_id)
        c = cursos_idx.loc[c_id]
        cats_tomadas.add(c.categoria)
        util = utilidad(u, c_id, nivel_ef, cats_tomadas)
        util_n = 1 / (1 + np.exp(-(util - 1.5)))           # utilidad normalizada a (0,1)
        encaja = NIV_NUM[c.dificultad] == nivel_ef

        eventos.append((u.user_id, c_id, t, "vista"))
        if rng.random() < 0.25:
            eventos.append((u.user_id, c_id, t + pd.Timedelta(hours=int(rng.integers(1, 48))), "favorito"))
        if rng.random() < 0.30 + 0.35 * util_n:
            t_ins = t + pd.Timedelta(hours=int(rng.integers(2, 96)))
            eventos.append((u.user_id, c_id, t_ins, "inscripcion"))
            p_fin = 0.45 + 0.25 * encaja - 0.15 * (c.duracion_horas > 100)
            if rng.random() < p_fin:
                semanas = c.duracion_horas / 6 * rng.uniform(0.7, 1.4)
                t_fin = t_ins + pd.Timedelta(days=max(1, int(semanas * 7)))
                nota = c.rating + rng.normal(0, 0.6) - (0 if encaja else 1.0)
                eventos.append((u.user_id, c_id, t_fin, "finalizacion", int(np.clip(round(nota), 1, 5))))
                completados += 1
                if completados % 2 == 0: nivel_ef = min(2, nivel_ef + 1)
        t = t + pd.Timedelta(days=max(1, int(rng.exponential(6))), hours=int(rng.integers(0, 24)))
    return eventos

filas = []
for u in usuarios.itertuples(index=False):
    filas.extend(simular_interacciones_usuario(u))

inter = pd.DataFrame([f if len(f) == 5 else (*f, np.nan) for f in filas],
                     columns=["user_id", "course_id", "timestamp", "evento", "calificacion"])
n_antes = len(inter)
inter = inter[inter["timestamp"] <= FECHA_CORTE].sort_values(["timestamp", "user_id"]).reset_index(drop=True)
PESO = {"vista": 1, "favorito": 2, "inscripcion": 3, "finalizacion": 4}
inter["peso_evento"] = inter["evento"].map(PESO)
inter.insert(0, "interaction_id", [f"I{i:05d}" for i in range(1, len(inter) + 1)])
print(f"Eventos generados: {n_antes} | eventos dentro del periodo observado: {len(inter)}")
display(inter["evento"].value_counts())
inter.head()

Eventos generados: 4024 | eventos dentro del periodo observado: 3680


,count
evento,
vista,1788
inscripcion,991
finalizacion,476
favorito,425


,interaction_id,user_id,course_id,timestamp,evento,calificacion,peso_evento
0,I00001,U097,C028,2026-01-07 18:00:00,vista,NaN,1
1,I00002,U080,C064,2026-01-11 19:00:00,vista,NaN,1
2,I00003,U097,C018,2026-01-13 03:00:00,vista,NaN,1
3,I00004,U097,C025,2026-01-14 10:00:00,vista,NaN,1
4,I00005,U077,C003,2026-01-14 15:00:00,vista,NaN,1


## 6. Validaciones de integridad

In [ ]:
chequeos = {
    "IDs de curso únicos": cursos["course_id"].is_unique,
    "IDs de usuario únicos": usuarios["user_id"].is_unique,
    "IDs de interacción únicos": inter["interaction_id"].is_unique,
    "Toda interacción apunta a un usuario existente": inter["user_id"].isin(usuarios["user_id"]).all(),
    "Toda interacción apunta a un curso existente": inter["course_id"].isin(cursos["course_id"]).all(),
    "Calificación solo en finalizaciones": inter.loc[inter["evento"] != "finalizacion", "calificacion"].isna().all(),
    "Calificación entre 1 y 5": inter["calificacion"].dropna().between(1, 5).all(),
    "Sin eventos antes del registro": (inter.merge(usuarios[["user_id", "fecha_registro"]], on="user_id")
                                        .pipe(lambda d: (d["timestamp"] >= pd.to_datetime(d["fecha_registro"])).all())),
    "Inscripción no ocurre sin vista previa": True,
}
# vista antes que inscripción para cada par usuario-curso
primero = inter.sort_values("timestamp").groupby(["user_id", "course_id"])["evento"].first()
chequeos["Inscripción no ocurre sin vista previa"] = (primero == "vista").all()

for k, v in chequeos.items(): print(("OK   " if v else "FALLA"), k)

pares = inter.groupby(["user_id", "course_id"]).size()
dispersion = 1 - len(pares) / (len(usuarios) * len(cursos))
resumen = pd.Series({
    "usuarios": usuarios.shape[0], "cursos": cursos.shape[0], "interacciones (eventos)": inter.shape[0],
    "pares usuario-curso distintos": len(pares),
    "pares con interacción positiva": inter[inter["evento"].isin(["inscripcion", "finalizacion"])]
                                        .groupby(["user_id", "course_id"]).ngroups,
    "dispersión de la matriz": round(dispersion, 3),
    "usuarios activos": inter["user_id"].nunique(), "cursos con al menos 1 interacción": inter["course_id"].nunique(),
})
resumen

OK    IDs de curso únicos
OK    IDs de usuario únicos
OK    IDs de interacción únicos
OK    Toda interacción apunta a un usuario existente
OK    Toda interacción apunta a un curso existente
OK    Calificación solo en finalizaciones
OK    Calificación entre 1 y 5
OK    Sin eventos antes del registro
OK    Inscripción no ocurre sin vista previa


,0
usuarios,150.000
cursos,107.000
interacciones (eventos),3680.000
pares usuario-curso distintos,1788.000
pares con interacción positiva,991.000
dispersión de la matriz,0.889
usuarios activos,150.000
cursos con al menos 1 interacción,107.000


## 7. Exportación

In [ ]:
cursos.to_csv(DATA / "cursos.csv", index=False, encoding="utf-8")
usuarios.to_csv(DATA / "usuarios.csv", index=False, encoding="utf-8")
inter.to_csv(DATA / "interacciones.csv", index=False, encoding="utf-8")
AFINIDAD.to_csv(DATA / "raw" / "regla_afinidad_objetivo_categoria.csv", encoding="utf-8")
for f in ["cursos.csv", "usuarios.csv", "interacciones.csv"]:
    print(f, "->", (DATA / f).stat().st_size // 1024, "KB")

cursos.csv -> 50 KB
usuarios.csv -> 18 KB
interacciones.csv -> 177 KB


### Limitaciones de este dataset (para discutir en el informe)
- Las interacciones son **simuladas**: las métricas que se obtengan miden qué tan bien cada algoritmo recupera los patrones definidos en las reglas, no el comportamiento de estudiantes reales.
- Las reglas favorecen la afinidad objetivo-categoría; un modelo de contenido o híbrido que use esas mismas variables tiene una ventaja estructural. Este sesgo se discutirá al comparar modelos.
- Se imputaron valores no publicados: nivel, duración y reseñas de los 8 cursos de edX en español, y rating de esos 8 cursos más los 9 de DataCamp.
- Las habilidades están en inglés para Coursera y en español para Platzi, DataCamp y edX. Esto afecta al modelo de contenido (TF-IDF), por lo que en el notebook de modelamiento se usará también la categoría como texto común a ambos idiomas.
- El número de reseñas no es del todo comparable entre plataformas.
- El catálogo es una fotografía al 19/10/2026; ratings y reseñas cambian con el tiempo.